# Colab 向けコードを VS Code で実行しやすいよう修正しました

このノートブックは元々 Google Colab 向けに書かれていたため、`google.colab.files` を使ったアップロード/ダウンロード処理をローカル実行向けに置換しました。

変更点:
- `google.colab` の削除。代わりに `tkinter.filedialog` でローカルから画像を選択します。
- 依存パッケージ（opencv-python, numpy, matplotlib, pillow）のインストール案内セルを追加しました。
- 出力はローカルに保存し、保存先パスを表示します。

使い方:
1. 必要であれば下の「依存パッケージ確認」セルを実行して不足パッケージをインストールしてください（セル内の `INSTALL = True` を使うか、ターミナルで `pip install opencv-python numpy matplotlib pillow` を実行します）。
2. その後、処理セルを実行するとファイル選択ダイアログが開きます。画像を選択すると透過 PNG が同じフォルダに `transparent_<元ファイル名>.png` として保存されます。

注意: Headless (GUI非対応) 環境では `tkinter` が使えないため、代わりにファイルパスリストを手動で渡す `process_files` を直接呼んでください。


In [9]:
# 依存パッケージ確認・インストールヘルパーセル（自動インストール有効）
import sys
import subprocess

required = {
    "cv2": "opencv-python",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "PIL": "pillow",
}

missing = []
for mod, pkg in required.items():
    try:
        if mod == "PIL":
            __import__("PIL")
        else:
            __import__(mod)
    except Exception:
        missing.append(pkg)

if missing:
    print("不足パッケージ:", missing)
    print("以下のコマンドでインストールを試みます（失敗した場合はターミナルで手動実行してください）：")
    print(f"{sys.executable} -m pip install --upgrade {' '.join(missing)}")

    def _install(pkgs):
        cmd = [sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs
        try:
            subprocess.check_call(cmd)
            print("インストール完了:", pkgs)
        except subprocess.CalledProcessError as e:
            print("インストールに失敗しました。以下のコマンドをターミナルで実行してください:")
            print(' '.join(cmd))
            print("エラー:", e)

    INSTALL = True  # 自動インストールをデフォルトで有効化
    if INSTALL:
        try:
            _install(missing)
        except Exception as e:
            print("自動インストール中に例外が発生しました:", e)
else:
    print("すべての依存パッケージはインストール済みです。")


すべての依存パッケージはインストール済みです。


In [10]:
# クロマキー処理 & ローカルファイル選択セル
import os
from io import BytesIO
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# OpenCV は主に高速処理に使うが、PIL/Numpy だけでも動きます
try:
    import cv2
except Exception:
    cv2 = None

# 高度なクロマキー関数（元コードをほぼ維持）
def advanced_chromakey(image_rgb, color_key='green', sensitivity=0.4, smoothness=0.1, spill_suppression=0.8):
    """
    プロフェッショナル品質のクロマキー処理を行う関数

    Args:
        image_rgb: RGB画像 (numpy array)
        color_key: 'green' または 'blue'
        sensitivity: キー色の検出感度 (0.0 - 1.0)。高いほど広く抜く。
        smoothness: エッジの滑らかさ (0.0 - 1.0)。
        spill_suppression: スピル除去の強度 (0.0 - 1.0)。
    """
    img_float = image_rgb.astype(np.float32) / 255.0
    r, g, b = img_float[:,:,0], img_float[:,:,1], img_float[:,:,2]

    if color_key == 'green':
        key_src = g - np.maximum(r, b)
        spill_val = np.maximum(r, b) * ((1.0 - spill_suppression) * 1.0 + spill_suppression * 1.1)
    else:
        key_src = b - np.maximum(r, g)
        spill_val = np.maximum(r, g) * ((1.0 - spill_suppression) * 1.0 + spill_suppression * 1.1)

    # マスク生成
    mask = (key_src - (0.5 - sensitivity)) / (smoothness + 1e-6)
    mask = 1.0 - np.clip(mask, 0.0, 1.0)

    # スピル除去
    img_despill = img_float.copy()
    if color_key == 'green':
        img_despill[:,:,1] = np.minimum(g, spill_val)
    else:
        img_despill[:,:,2] = np.minimum(b, spill_val)

    # 合成: RGBA
    result = np.dstack((img_despill, mask))
    result = (np.clip(result, 0.0, 1.0) * 255).astype(np.uint8)
    return result


def process_files(file_paths, color_key='green', sensitivity=0.45, smoothness=0.05, spill_suppression=0.9, show_preview=True):
    """ 選択したファイルを順に処理して保存する """
    for path in file_paths:
        print(f"処理中: {path}")
        image = Image.open(path).convert('RGB')
        img_np = np.array(image)

        result_image = advanced_chromakey(
            img_np,
            color_key=color_key,
            sensitivity=sensitivity,
            smoothness=smoothness,
            spill_suppression=spill_suppression
        )

        out_name = f"transparent_{os.path.splitext(os.path.basename(path))[0]}.png"
        out_path = os.path.join(os.path.dirname(path), out_name)
        Image.fromarray(result_image).save(out_path)
        print(f"保存: {out_path}")

        if show_preview:
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(img_np)
            plt.title("Original")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(result_image)
            plt.title("Processed (Transparent)")
            plt.axis('off')
            plt.show()


# ファイル選択 UI（ローカル実行向け）
def select_and_run():
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as e:
        print("tkinter を使えません: GUI 環境で実行してください。もしくはファイルパスのリストを process_files に渡してください。", e)
        return

    root = tk.Tk()
    root.withdraw()
    file_paths = filedialog.askopenfilenames(
        title="画像を選択",
        filetypes=[("画像ファイル", "*.png *.jpg *.jpeg *.bmp *.tif *.tiff")]
    )
    if not file_paths:
        print("ファイルが選択されませんでした。")
        return

    process_files(file_paths)


# ノートブックで簡単に使えるように、ユーザーはこの関数を実行してください。
# select_and_run()  を実行するとダイアログが開き、選択した画像が処理・保存されます。


# 使い方（ワンステップ）
このセルを実行すると、ファイル選択ダイアログが開き、選択した画像が順に処理・保存されます。

実行手順:
1. 上の「依存パッケージ確認・インストールヘルパー」セルを先に実行して、必要なパッケージをインストールしてください。
2. このセルを実行してください（GUI環境が必要です）。

注意: サーバやヘッドレス環境では GUI が使えないため、代わりに `process_files([...])` または `process_folder(...)` を使ってください。


In [11]:
# ワンステップ実行セル
try:
    select_and_run()
except Exception as e:
    print("select_and_run() 実行中にエラー:", e)
    print("GUI が使えない場合は、process_files([...]) か process_folder(path) を使ってください。")


select_and_run() 実行中にエラー: no display name and no $DISPLAY environment variable
GUI が使えない場合は、process_files([...]) か process_folder(path) を使ってください。


In [12]:
# フォルダ内の画像を一括処理するヘルパー
import glob
import os

def process_folder(folder_path, recursive=False, **kwargs):
    """指定フォルダ内の画像を一括処理します。

    Args:
        folder_path: 対象フォルダのパス
        recursive: サブフォルダを再帰的に検索するか
        **kwargs: process_files に渡す追加パラメータ（color_key 等）
    """
    pattern = "**/*" if recursive else "*"
    exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    files = [p for p in glob.glob(os.path.join(folder_path, pattern), recursive=recursive) if p.lower().endswith(exts)]
    if not files:
        print("画像が見つかりませんでした:", folder_path)
        return
    process_files(files, **kwargs)

# 使い方例（実行時はコメント解除してパスを変更してください）
# process_folder(r"C:\path\to\images", recursive=False, show_preview=True)


In [14]:
# テスト実行セル: 指定ファイルを処理します（リポジトリ内の候補パスを追加）
import os
from pathlib import Path

test_name = "Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png"
# 候補パスを順にチェック（カレント、ワーキングディレクトリ、プロジェクト内のdocパスなど）
candidates = [
    test_name,
    os.path.join(os.getcwd(), test_name),
    str(Path.cwd() / test_name),
    os.path.join(os.getcwd(), "doc", "02_Characters", "images", "player", test_name),
    str(Path.cwd() / "doc" / "02_Characters" / "images" / "player" / test_name),
]
path = None
for p in candidates:
    if os.path.exists(p):
        path = p
        break

if path is None:
    print("ファイルが見つかりませんでした。候補パス:")
    for c in candidates:
        print("  ", c)
    print("カレントディレクトリ:", os.getcwd())
    print("作業ディレクトリを確認するか、絶対パスを指定して `process_files(['絶対パス'])` を呼んでください。")
else:
    print("処理対象ファイル:", path)
    try:
        process_files([path], show_preview=True)
        out_name = f"transparent_{os.path.splitext(os.path.basename(path))[0]}.png"
        out_path = os.path.join(os.path.dirname(path), out_name)
        print("出力ファイル:", out_path)
    except Exception as e:
        print("処理中にエラーが発生しました:", e)
        print("依存パッケージの確認・カーネルの選択を行ってください。")

ファイルが見つかりませんでした。候補パス:
   Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png
   /content/Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png
   /content/Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png
   /content/doc/02_Characters/images/player/Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png
   /content/doc/02_Characters/images/player/Gemini_Generated_Image_6iwkfu6iwkfu6iwk.png
カレントディレクトリ: /content
作業ディレクトリを確認するか、絶対パスを指定して `process_files(['絶対パス'])` を呼んでください。
